## Status (updated 2026-07-31) -- READ BEFORE RUNNING

Fixed since the original version: (1) modularity is now always scored against one canonical
ARBF-on-PCA graph, not whatever graph happened to be sitting on `ad.obsp['connectivities']`
last; (2) the `AnnData.__init__` dtype monkeypatch is guarded against re-run recursion.
Helper functions now live in `interpretable_ssl/evaluation/batch_correct_baselines.py`
(imported by the "Helper functions" cell below) instead of being defined inline here.

**Current run scope (set in the Config cell):** `CORRECTION_METHODS = ['harmony',
'scvi_gauss']`, `RUN_SEACELLS = True` -- scProto vs. {Harmony, scVI (Gaussian)} x
{SEACells, Leiden}. Plain ZINB `scvi` is deliberately left OUT of the display as of
2026-07-31 (having both scVI variants in every table at once was more confusing than
useful) -- its results are untouched on disk (`seacell_X_scvi` / `leiden_X_scvi_K{K}`)
and can be brought back any time by adding `'scvi'` back to `CORRECTION_METHODS`.
`harmony` already has a complete cached run on disk for all 3 RNA-seq datasets (verified
2026-07-30) -- with `SKIP_IF_EXISTS = True` it's a cheap metrics.json/cache reload, no
retraining. **`scvi_gauss` is the one thing this run actually computes** -- real scVI
model training with `gene_likelihood='normal'` instead of the ZINB default. `stage1z`
and `bbknn` are left out of `CORRECTION_METHODS` for now (stage1z's own results are
already available separately; bbknn's install is still unreliable, see pipInstall01) --
widen `CORRECTION_METHODS` to bring any of these back.

**Why `scvi_gauss` exists (2026-07-31):** scProto's Stage-1 pretrain uses `recon_loss='mse'`
on log-normalized expression (`defaults.py`), while plain scVI reconstructs raw counts under
a ZINB likelihood -- two different noise-model families, not just two different
architectures, which confounds "is scVI's batch-correction mechanism better" with "is ZINB
just a better-suited loss for scRNA-seq counts than MSE." `scvi_gauss` swaps in
`gene_likelihood='normal'` (Gaussian NLL, learned per-gene variance -- reduces to weighted
MSE) -- still scVI's real, unmodified architecture (same encoder, same
library-size-scaled/softmax decoder mean, same batch conditioning, same n_latent=8, same
epoch budget), just without the NB/zero-inflation shape. This is NOT a perfect match to
scProto's plain MSE-on-lognorm -- the decoder mean is still `library_size * softmax(...)`,
not an unconstrained prediction directly against log-normalized targets (stock scVI has no
supported way to bypass its library-size machinery) -- but it removes the single biggest
source of mismatch. Report it as "scVI (Gaussian)", not "scVI (MSE)". `stage1z`
(scProto's own actual MSE-on-lognorm Stage-1 encoder) remains the more literal MSE control
and needs no new training -- see its own results elsewhere in this notebook / the paper's
existing runs.

Data preprocessing for scVI (both variants): trains on `adata.layers['counts']` (verified
2026-07-31 to be genuine pre-normalization raw counts for all 3 datasets, not a copy of the
log-normalized `.X` -- see `add_scvi_emb`'s docstring), NOT `ad.X` (which is
log-normalized at the point this notebook's `ad` reaches `run_correction_method` --
`get_stage1_latent`/`get_preprocessed_adata` both load with `use_counts=False`). scVI's own
internal library-size normalization is what it uses raw counts for; feeding it
already-normalized data would double-normalize and break that machinery, hence the
raw-counts layer swap inside `add_scvi_emb` regardless of `gene_likelihood`.

Folder/cache naming: `scvi_gauss` never collides with plain `scvi` -- distinct tag
(`X_scvi_gauss` vs `X_scvi`), distinct cache files (`_cache_X_scvi_gauss_*` vs
`_cache_X_scvi_*`), distinct result folders (`seacell_X_scvi_gauss` /
`leiden_X_scvi_gauss_K{K}` vs `seacell_X_scvi` / `leiden_X_scvi_K{K}`). Both are 8-dim
(`n_latent=8`, matching scProto's own `latent_dims`), and both go through the exact same
downstream SEACells/Leiden pipeline (same K-matching, same canonical-modularity recompute,
same result tables below) as every other correction method here -- nothing method-specific
needed beyond `run_correction_method`'s new `scvi_gauss` branch.

If you already have this notebook open in another Colab tab, close it (or File > Revert)
first -- Colab autosaving from a stale tab can silently overwrite this file. Also note:
`rare_affinity_purity_{ds}.json` (the embedding-only rare-cell diagnostic a few cells down)
is a file SHARED with sibling baseline notebooks (e.g. `combat_then_cluster_baselines.ipynb`)
-- each overwrites it wholesale with whatever it computed, so if another such notebook runs
after this one, `scvi_gauss`'s row there can disappear even though its actual SEACells/Leiden
results (which live in their own per-method folders, never shared) are unaffected.


# Rebuttal experiment E1: batch-correct-then-cluster baselines -- {scPoli, Harmony, scVI, BBKNN} latent -> {SEACells, Leiden}

**Current default run scope:** the Config cell below currently sets `CORRECTION_METHODS = ['harmony', 'scvi', 'scvi_gauss']` and `RUN_SEACELLS = True` -- scProto vs. {Harmony, scVI (ZINB), scVI (Gaussian)} x {SEACells, Leiden}. `stage1z` and `bbknn` are left out for now (see Status cell above) -- widen `CORRECTION_METHODS` to bring them back.

**This is the single, merged notebook for E1.** It used to be split into two
notebooks (`e1_two_step_baseline_1_scpoli.ipynb` handling scPoli alone, `e1_two_step_baseline_2_harmony_scvi_bbknn.ipynb`
handling the other three) purely because scVI/Harmony/BBKNN need heavier installs -- that split caused real
maintenance risk (every helper function was duplicated near-verbatim across both files, so a bugfix in one could
silently not make it into the other -- this is exactly what happened with the Leiden resolution-search bug). Both
old notebooks are kept on disk for now as an execution-history archive but are superseded by this one; delete them
once this notebook's outputs are trusted.

**Purpose.** Reviewer F5RB (Q1): *"Please compare scProto with stronger two-step baselines, such as scPoli latent + SEACells, scVI
latent + SEACells, Harmony latent + SEACells, and BBKNN graph + SEACells."* e9Ho: *"K-means handicaps the baselines ... the
community-structure win is confounded with the clusterer, not the embedding."* This notebook runs exactly that: for each of
**4 batch-correction methods** (scPoli Stage-1, Harmony, scVI, BBKNN), get the corrected embedding/graph, then run
**2 graph-aware clustering methods** (SEACells, Leiden) on it -- replacing the paper's current weak `scPoli + K-means` baseline
with proper two-step baselines. In other words: does "correct the batch effect first, then find metacells" (the reviewers'
suggested alternative) work as well as scProto's joint approach?

**Per RNA-seq dataset (pancreas, lung, pbmc-immune), per correction method:**
- **scPoli Stage-1**: load the *existing* pretrain checkpoint (no retraining), encode cells -> latent `X_stage1z`.
- **Harmony**: PCA + `harmonypy.run_harmony` directly (pinned harmonypy==0.0.9; not routed through `scanpy.external.pp.harmony_integrate`, which hardcodes an output shape that silently breaks under newer harmonypy versions) -> `X_harmony`.
- **scVI**: trains a fresh scVI model (via this codebase's `embedding_metrics.add_scvi_emb`) -> `X_scvi`. Slower than the other methods -- it's actual model training, not a lookup. Two variants: plain `scvi` (scvi-tools' default ZINB reconstruction likelihood) and `scvi_gauss` (`gene_likelihood='normal'`, a Gaussian/MSE-like reconstruction on the same raw-count target, loss-matched closer to scProto's own `recon_loss='mse'` Stage-1 pretrain -- see the Status cell above for the full rationale/caveats). Both are 8-dim, matching scProto's latent.
- **BBKNN**: `scanpy.external.pp.bbknn` -> a batch-balanced kNN graph directly (no embedding step -- this graph itself is the affinity fed to SEACells/Leiden below).

For the embedding-based methods, **one** adaptive-RBF affinity graph is built per embedding (same construction SEACells/scProto
use by default -- see `method.tex` / `results.tex`: *"adaptive Gaussian kernel on PCA embeddings"*, here pointed at the corrected
embedding instead of PCA) and fed to **both** SEACells and Leiden, so any SEACells-vs-Leiden gap is purely the clustering algorithm,
never confounded with a different graph construction (directly answers e9Ho's "confounded with clusterer, not embedding").

**Leiden hits target K exactly.** `leiden_resolution_search` over-segments (pushes resolution up until the partition has
more communities than target K -- always achievable) then greedily merges whichever pair of clusters loses the LEAST
modularity, until exactly K clusters remain (same principle Louvain/Leiden's own multilevel algorithm uses internally to
build its hierarchy). An earlier version used plain resolution bisection with a fixed upper bound and silently landed at
~70-90 clusters when the target was 220-300 -- invalidating any modularity comparison, since modularity is not
resolution-invariant (fewer/larger clusters score higher on a fixed graph almost by construction). Do not reintroduce that
bug; always verify `n_clusters` in the saved metrics.json matches `target_k` before trusting a Leiden modularity number.

**Metrics: only what the paper already reports** -- no new diagnostics, except DGE consistency is turned off
(`compute_dge=False` everywhere) since it is not needed for this hypothesis test and was failing on non-lognormalized
data. `compute_task1_metrics` / `calc_task2_metrics` (same functions the paper's existing numbers come from -- see
`method.tex`'s community/nassoc/usage/recon losses section) give Modularity / Batch Entropy / Purity (Table 1) and
Coverage / scGraph (Table 2); `rare_celltype_purity_table` gives the per-batch rare-cell Coverage / Homogeneity
(`results.tex` Table 2, tab:rare_cells).

**On the modularity number specifically (e9Ho's point on this):** modularity here is scored against the
adaptive-RBF-on-PCA graph, which is the same graph scProto's `L_community` is trained to match every step. None of the
two-step baselines in this notebook get that training-time guidance -- they cluster a graph built from a *different*
embedding (scPoli/Harmony/scVI latent) that only gets scored against the PCA graph after the fact. That is not a
causally fair comparison to scProto, and should not be presented as one. Treat this modularity number as a plausibility
sanity check only (if a baseline that never sees the PCA graph can match or beat scProto's modularity against it, that
would already be surprising and worth flagging) -- not as the load-bearing evidence. The actual evidence for "does
batch-correct-then-cluster work as well as scProto" has to come from metrics no method gets a training-time advantage
on: cell-type purity, batch entropy, and especially the rare-cell coverage/homogeneity table, all computed identically,
post-hoc, for every method regardless of what it optimized during training.

**Before running:** assumes Stage-1 pretrain checkpoints already exist for pancreas / lung / pbmc-immune (`cvae_epochs=50`,
`batch_size=1024`, see `train_scproto.ipynb`) -- this notebook does not retrain Stage 1, only loads it.

**Outputs, per dataset, per method:**
- `MODEL_DIR/{ds}/seacell_X_{method}/` -- SEACells-on-{method} (metrics.json, umap_cells.csv, umap_protos.csv, ...)
- `MODEL_DIR/{ds}/leiden_X_{method}_K{K}/` -- Leiden-on-{method} (same file set)

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 185.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 88.4 MB/s eta 0:00:00
   ━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 111.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 whi

In [1]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [2]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test. No anndata.io/anndata.abc bridge needed anymore (removed -- that only
# existed to patch an old, explicitly-pinned anndata==0.10.6; the current install
# lets anndata resolve naturally to whatever scvi-tools/scarches need, which already
# has both natively). bbknn excluded here -- temporarily removed, see pipInstall01.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)


/tmp/ipykernel_1839/2506724420.py:20: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ver = getattr(mod, '__version__', '?')
/tmp/ipykernel_1839/2506724420.py:20: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  ver = getattr(mod, '__version__', '?')


  OK   anndata          (import anndata, version 0.13.2)
  OK   scanpy           (import scanpy, version 1.12.3)


  FAIL scarches         (import scarches): ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)

1 package(s) failed to import: ['scarches'] -- re-run that package's specific pip install line above and check its full error output before proceeding.


In [3]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [4]:
# Extra imports not already covered by nb_setup.py (which already pulls in
# run_mc_task, eval_seacell_task1/2/3, load_task1_multi, show_table, clean_run_names,
# rare_celltype_purity_table, TASK1_METRICS, TASK2_METRICS via
# `from interpretable_ssl.evaluation.paper_figures import *` and
# `from interpretable_ssl.evaluation.metric_helpers.result_tables import *`).
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir

print("extra imports ready")


extra imports ready


## Config

Same hyperparameters `train_scproto.ipynb` used to produce the existing Stage-1 checkpoints for these three datasets
(`cvae_epochs=50`, `batch_size=1024`) -- must match, or `load_pretrain_checkpoint()` looks in the wrong folder. `K` (n_SEACells /
target Leiden cluster count) defaults to each dataset's `num_prototypes` from `DATASETS`, matching `results.tex`:
*"All baselines are configured to produce the same number of metacells K as scProto."*


In [5]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']

SKIP_IF_EXISTS = True  # skip a baseline entirely (embedding + SEACells + Leiden) if its
                        # metrics.json already exists on disk -- re-running the dataset
                        # cell after a crash/interrupt never redoes already-saved work
                        # (matters most for scVI, the slow one: real model training).

# 'bbknn' TEMPORARILY REMOVED -- see pipInstall01's comment (suspected stale numpy/
# scipy pin in the bbknn package itself). Re-add once installable again.
# 'scvi' (plain ZINB) REMOVED FROM DISPLAY 2026-07-31 -- its results are still fully
# intact on disk (seacell_X_scvi / leiden_X_scvi_K{K}, untouched by this change) and
# can be brought back into the tables any time by adding 'scvi' below again; it's just
# not shown alongside 'scvi_gauss' right now since having both scVI variants in every
# table at once was more confusing than useful. 'scvi_gauss' (Gaussian likelihood,
# loss-matched closer to scProto's own MSE Stage-1 -- see Status cell) is the one
# scVI variant actually being compared against going forward.
# Full sweep -- restore this to also rerun stage1z / bring ZINB scVI back:
# CORRECTION_METHODS = ['stage1z', 'harmony', 'scvi', 'scvi_gauss']
CORRECTION_METHODS = ['harmony', 'scvi_gauss']
# -- 'harmony' already has a complete cached seacell_X_harmony_d8 / leiden_X_harmony_d8_K{K}
#    run on disk for all 3 RNA-seq datasets (verified 2026-07-30) -- with
#    SKIP_IF_EXISTS=True below, re-running it is just a metrics.json / cached-embedding
#    reload, NOT retraining. Nothing to do for it, it's here only so the result tables
#    below include it.
# -- 'scvi_gauss' is the one real compute here: scVI trained with gene_likelihood='normal'
#    (Gaussian reconstruction) instead of the ZINB default -- loss-matched closer to
#    scProto's own recon_loss='mse' Stage-1 pretrain (see the Status cell at the top of
#    this notebook for the full rationale/caveats: it removes the NB/zero-inflation
#    likelihood shape but is still library-size/softmax-coupled, so it's "scVI
#    (Gaussian)", not a literal MSE-on-lognorm match -- stage1z remains the more literal
#    MSE control). Same n_latent=8, same epoch budget, same reference/query split as the
#    ZINB run -- only gene_likelihood differs. Saved under its own tag (X_scvi_gauss),
#    never collides with the still-intact-on-disk plain-scvi (ZINB) cache/results.
METHOD_DISPLAY_NAMES = {
    'stage1z': 'scPoli (Stage-1)',
    'harmony': 'Harmony',
    'scvi': 'scVI',
    'scvi_gauss': 'scVI (Gaussian)',
    'bbknn': 'BBKNN',
}

RUN_SEACELLS = True  # also compute/load SEACells-on-{method}, not just Leiden.


## Helper functions

Moved to `interpretable_ssl/evaluation/batch_correct_baselines.py` (imported below) --
this notebook now only holds config + the per-dataset run/results cells. The module also
carries the two environment patches (SEACells' stray `dtype=` kwarg, Arrow-backed string
columns breaking the h5ad writer) that used to be separate cells here; they're applied
automatically on import.


In [6]:
from interpretable_ssl.evaluation.batch_correct_baselines import run_all_baselines_for_dataset

print("baseline helper functions ready (interpretable_ssl.evaluation.batch_correct_baselines)")


baseline helper functions ready (interpretable_ssl.evaluation.batch_correct_baselines)


## Run: Pancreas

In [7]:
pancreas_results = run_all_baselines_for_dataset(
    'pancreas', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS,
)


loading pancreas data
✅ Already subsetted to HVGs (4000 genes).


 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=500 → 512000 edges/epoch (of 1155146 total

  0%|          | 0/16 [00:00<?, ?it/s]


=== [pancreas] batch-correction method: harmony ===
[pancreas] harmony_d8_k100: reusing cached embedding at /content/drive/MyDrive/models/pancreas/_cache_X_harmony_d8_k100_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/16382 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/16382 [00:00<?, ?it/s]

Constructing CSR matrix...
[pancreas] scProto (d=8): 0.454 +/- 0.184 (n=8 batches)
[pancreas] affinity purity saved to /content/drive/MyDrive/models/pancreas/rare_affinity_purity_pancreas.json (merged with any existing entries from other runs/notebooks)


## Run: Lung

In [8]:
lung_results = run_all_baselines_for_dataset(
    'lung', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS,
)


loading lung data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading lung
loading lung data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [16]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.255/24.467/119.300, effk_med=63.8, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/lung/pretrain/pretrain_ds-lung_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'lung', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'batch'}
📊 EdgeDataset: 2447924 edges
   Weight range: [0.0137, 0.9491]
   umap_steps_per_epoch=500 →

  0%|          | 0/32 [00:00<?, ?it/s]


=== [lung] batch-correction method: harmony ===
[lung] harmony_d8_k100: reusing cached embedding at /content/drive/MyDrive/models/lung/_cache_X_harmony_d8_k100_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/32472 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/32472 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/32472 [00:00<?, ?it/s]

Constructing CSR matrix...
[lung] scProto (d=8): 0.586 +/- 0.208 (n=15 batches)
[lung] affinity purity saved to /content/drive/MyDrive/models/lung/rare_affinity_purity_lung.json (merged with any existing entries from other runs/notebooks)


## Run: PBMC (Immune)

In [9]:
immune_results = run_all_baselines_for_dataset(
    'pbmc-immune', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS,
)


loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading pbmc-immune
loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [5]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=1.667/25.923/299.545, effk_med=62.9, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pbmc-immune/pretrain/pretrain_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pbmc-immune', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'study'}
📊 EdgeDataset: 2590828 edges
   Weight range: [0.0050, 0.9224]
   umap_ste

  0%|          | 0/33 [00:00<?, ?it/s]


=== [pbmc-immune] batch-correction method: harmony ===
[pbmc-immune] harmony_d8_k100: reusing cached embedding at /content/drive/MyDrive/models/pbmc-immune/_cache_X_harmony_d8_k100_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/33506 [00:00<?, ?it/s]

Constructing CSR matrix...
[pbmc-immune] scProto (d=8): 0.849 +/- 0.085 (n=5 batches)
[pbmc-immune] affinity purity saved to /content/drive/MyDrive/models/pbmc-immune/rare_affinity_purity_pbmc-immune.json (merged with any existing entries from other runs/notebooks)


## Results comparison

Pulls in **every** existing run under `MODEL_DIR/{ds}/*` (scProto, SEACells(PCA), MetaQ, scPoli-cVAE, UMAP, ...) alongside the
8 new baselines computed above (4 correction methods x {SEACells, Leiden}) -- `load_task1_multi` / `rare_celltype_purity_table`
just scan folders + `metrics.json` / `umap_cells.csv`, they don't need to know these runs are new.

Keyword note: SEACells folders are `seacell_X_{method}`, Leiden folders are `leiden_X_{method}_K{n}` -- the full
`seacell_X_{method}` / `leiden_X_{method}` strings are used as match keywords below (not just `X_{method}`) so a SEACells run
can never accidentally substring-match its sibling Leiden run's folder name, or vice versa.

**Reminder (see intro cell):** the modularity column below is scored against the arbf-on-PCA graph that only scProto is
trained to match -- treat it as a plausibility check, not the primary evidence. Purity, batch entropy, and especially the
rare-cell coverage/homogeneity table (two cells down) are the fair, post-hoc comparison.

### Harmony + Leiden -- realized cluster count vs. scProto's target K

`leiden_resolution_search` targets scProto's own `num_prototypes` for this dataset exactly
(see intro cell) -- this checks it actually landed there for the Harmony run, and would
surface a mismatch if the resolution search's `max_res` cap were ever hit instead (see
`leiden_resolution_search`'s docstring in `batch_correct_baselines.py`). Reads directly
from each run's saved `metrics.json` (`n_clusters` / `resolution` fields written by
`run_leiden_on_latent`), so this works whether the run above was fresh or a
`skip_if_exists` cache hit -- and simply shows nothing for a dataset that hasn't been run yet.


In [10]:
target_k = {ds: DATASETS[ds]['num_prototypes'] for ds in RNA_SEQ_DATASETS}

df_harmony_k = load_task1_multi(RNA_SEQ_DATASETS, metrics=['n_clusters', 'resolution'])
if df_harmony_k.empty:
    print("No runs found yet under MODEL_DIR for RNA_SEQ_DATASETS -- run the 'Run: ...' "
          "cells above first.")
else:
    is_harmony_leiden = df_harmony_k.index.get_level_values('run').str.startswith('leiden_X_harmony')
    df_harmony_k = df_harmony_k[is_harmony_leiden].copy()
    if df_harmony_k.empty:
        print("No leiden_X_harmony run found yet for any dataset in RNA_SEQ_DATASETS -- "
              "run the 'Run: ...' cells above first (with 'harmony' in CORRECTION_METHODS).")
    else:
        df_harmony_k['target_k'] = [target_k[ds] for ds, _run in df_harmony_k.index]
        df_harmony_k['matches_target'] = df_harmony_k['n_clusters'] == df_harmony_k['target_k']
        display(df_harmony_k)


n_clusters resolution  target_k  \
dataset     run                                                        
pancreas    leiden_X_harmony_K220         220.0       32.0       220   
            leiden_X_harmony_d8_K220      220.0       32.0       220   
lung        leiden_X_harmony_K300         300.0       64.0       300   
            leiden_X_harmony_d8_K300      300.0       64.0       300   
pbmc-immune leiden_X_harmony_K119         119.0   7.992236       300   
            leiden_X_harmony_K300         300.0       16.0       300   
            leiden_X_harmony_d8_K300      300.0       32.0       300   

                                      matches_target  
dataset     run                                       
pancreas    leiden_X_harmony_K220               True  
            leiden_X_harmony_d8_K220            True  
lung        leiden_X_harmony_K300               True  
            leiden_X_harmony_d8_K300            True  
pbmc-immune leiden_X_harmony_K119              False  
            leiden_X_harmony_K300               True  
            leiden_X_harmony_d8_K300            True

### SEACells -- realized metacell count vs. scProto's target K (read-only, no recompute)

SEACells folders don't encode K in their name the way Leiden's do (`seacell_X_harmony`,
not `seacell_X_harmony_K300`), so a stale run from an old `num_prototypes` value can't be
spotted from the folder listing alone, and `metrics.json`'s own `K_target` field gets
silently overwritten to the current target on every `skip_if_exists` cache hit even for a
stale run -- see `get_realized_seacell_count`'s docstring in `batch_correct_baselines.py`.
This reads the actual realized count from each run's saved `cell_assignments.csv`
directly -- no archetypal analysis is re-run, this is purely a check of what's already on
disk. `RUN_SEACELLS` above only controls whether new SEACells runs execute; this cell
works regardless of that setting, checking whatever the last SEACells run for each
(dataset, method) actually produced.


In [11]:
from interpretable_ssl.evaluation.batch_correct_baselines import get_realized_seacell_count

seacell_k_rows = []
for ds_id in RNA_SEQ_DATASETS:
    for method in CORRECTION_METHODS:
        n_actual = get_realized_seacell_count(ds_id, f'X_{method}')
        k = target_k[ds_id]
        seacell_k_rows.append({
            'dataset': ds_id, 'method': method,
            'n_actual': n_actual, 'target_k': k,
            'matches_target': (n_actual is not None and abs(n_actual - k) <= 0.05 * k),
        })

df_seacell_k = pd.DataFrame(seacell_k_rows).set_index(['dataset', 'method'])
if df_seacell_k['n_actual'].isna().all():
    print("No seacell_X_{method} runs found on disk yet for RNA_SEQ_DATASETS/CORRECTION_METHODS.")
else:
    display(df_seacell_k)


n_actual  target_k  matches_target
dataset     method                                        
pancreas    harmony          220       220            True
            scvi_gauss       220       220            True
lung        harmony          300       300            True
            scvi_gauss       300       300            True
pbmc-immune harmony          300       300            True
            scvi_gauss       300       300            True

In [12]:
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# scProto runs verified against the paper's published numbers via generate_tables.ipynb
# (the actual notebook used to build the paper's tables -- it dumps every candidate run
# for manual reading, no auto-selection) and cross-checked against appendix/training.tex's
# described loss weights/hyperparameters -- these are NOT just "close", they're
# bit-identical to the published numbers on multiple metrics simultaneously per dataset.
SCPROTO_CANONICAL_RUNS = {
    'proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
}

# BUG FIX 2026-07-30: load_task1_df already runs every folder name through
# extract_model_key() (strips _ds-panc/_NP220/_cvae_e50/_v31 etc. so the SAME
# experiment aligns as one row across datasets) BEFORE this notebook ever sees the
# `run` index -- so matching against the raw folder names above (as this used to do)
# never matched anything, and scProto silently never appeared in Table 1/2. All three
# raw names normalize to one shared stripped key (that's the point of
# extract_model_key) -- computed here instead of re-hardcoded, and asserted so a
# future change to extract_model_key's patterns fails loudly instead of silently
# reintroducing this bug.
SCPROTO_KEY = extract_model_key(next(iter(SCPROTO_CANONICAL_RUNS)))
assert all(extract_model_key(r) == SCPROTO_KEY for r in SCPROTO_CANONICAL_RUNS), (
    "SCPROTO_CANONICAL_RUNS entries no longer normalize to one shared key -- "
    "extract_model_key's stripping patterns changed; update the scProto matching logic."
)

# One shared {run_prefix: display_name} map for scProto + SEACells(PCA) + one
# {SEACells, Leiden} entry per method currently in CORRECTION_METHODS -- built once
# here, reused by Table 1, Table 2, and the rare-cell-type table below (previously
# duplicated in each of those cells separately).
MODEL_KEYWORDS = {SCPROTO_KEY: 'scProto'}
MODEL_KEYWORDS['seacell'] = 'SEACells (PCA)'

# 'harmony' excluded from this generic loop on purpose: run_correction_method now
# corrects Harmony at scProto's own latent dimension instead of the usual 50 (see
# its harmony_n_comps docstring), so its folders are 'seacell_X_harmony_d{n}' /
# 'leiden_X_harmony_d{n}_K{K}', not the plain 'seacell_X_harmony' /
# 'leiden_X_harmony_K{K}' this loop would generate. One Harmony baseline, added
# explicitly below with its real (dimension-qualified) folder name -- not two.
for _method in CORRECTION_METHODS:
    if _method == 'harmony':
        continue
    _disp = METHOD_DISPLAY_NAMES[_method]
    MODEL_KEYWORDS[f'seacell_X_{_method}'] = f'SEACells ({_disp})'
    MODEL_KEYWORDS[f'leiden_X_{_method}']  = f'Leiden ({_disp})'

# HARMONY_DIM must match scProto's actual latent_dims (currently 8 for all three
# datasets here, per the Stage-1 pretrain params printed in the 'Run:' cells above)
# -- update this if that ever changes, since MODEL_KEYWORDS below is a literal
# string match, not computed dynamically like the .py code is.
HARMONY_DIM = 8
if 'harmony' in CORRECTION_METHODS:
    MODEL_KEYWORDS[f'seacell_X_harmony_d{HARMONY_DIM}'] = 'SEACells (Harmony)'
    MODEL_KEYWORDS[f'leiden_X_harmony_d{HARMONY_DIM}']  = 'Leiden (Harmony)'

def _keep_and_rename_runs(df, model_keywords=MODEL_KEYWORDS):
    """Filter load_task1_multi's output down to just the runs in model_keywords, and
    collapse each method's per-dataset '_K{n}' folder-name suffix (e.g.
    'leiden_X_harmony_K300' for one dataset vs '..._K220' for another, since target K
    is scProto's num_prototypes and differs by dataset) into ONE shared display row per
    method -- otherwise Table 1/2 fragment into a separate row per distinct K value
    instead of one compact row per method spanning all datasets.

    If more than one on-disk run for the same dataset maps to the same display name
    (e.g. a stale leiden_X_harmony_K* folder left over from an earlier num_prototypes
    value, alongside the current one), the extra row is dropped with a WARNING rather
    than silently picked or crashing show_table's unstack() on the duplicate index --
    investigate and clean up the stale run directory if you see that warning.
    """
    runs = df.index.get_level_values('run')
    datasets = df.index.get_level_values('dataset')
    stripped = runs.str.replace(r'_K\d+$', '', regex=True)
    keep_mask = stripped.isin(model_keywords)

    kept_runs, kept_datasets = runs[keep_mask], datasets[keep_mask]
    kept_display = stripped[keep_mask].map(model_keywords)

    out = df[keep_mask].copy()
    out.index = pd.MultiIndex.from_arrays([kept_datasets, kept_display], names=['dataset', 'run'])

    dupe_mask = out.index.duplicated(keep='first')
    if dupe_mask.any():
        stale = list(zip(kept_datasets[dupe_mask], kept_runs[dupe_mask], kept_display[dupe_mask]))
        print(f"WARNING: dropped {dupe_mask.sum()} duplicate (dataset, method) row(s) -- "
              f"likely a stale run at an old K value still on disk. "
              f"(dataset, on-disk folder, display name): {stale}")
        out = out[~dupe_mask]
    return out

# --- Table 1 (community structure / batch integration): modularity, batch entropy, purity ---
df_task1 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK1_METRICS)
df_task1 = _keep_and_rename_runs(df_task1)
show_table(df_task1, metrics=TASK1_METRICS, dataset_display_names=dataset_display_names)


In [13]:
# --- Table 2 (metacell representation quality): coverage, scGraph (DGE consistency turned off, see intro) ---
# Filtered/renamed the same way as Table 1 -- see _keep_and_rename_runs (defined in the Table 1 cell above).
df_task2 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK2_METRICS)
df_task2 = _keep_and_rename_runs(df_task2)
show_table(df_task2, metrics=TASK2_METRICS, dataset_display_names=dataset_display_names)


In [14]:
CORRECTION_METHODS

['harmony', 'scvi_gauss']

In [15]:
# --- Rare-cell-type table (the key hypothesis test): coverage + homogeneity + F1, same
# per-batch-rare-cell definition results.tex uses for Table 2 (tab:rare_cells) ---
#
# scProto keyword: three dataset-specific exact folder names, all mapped to
# display name 'scProto'. Verified:
# the computed numbers now match the paper's published Table 2 exactly (pancreas
# 0.66/0.54, lung 0.72/0.60, pbmc 1.00/0.86 -- coverage/homogeneity, bit-identical).
#
# batch_rare_f1_macro_mean added -- per-rare-type F1 (precision = purity of that type's
# dedicated metacell(s), recall = fraction of that type's cells that landed in one),
# macro-averaged across rare types, then mean+-std across batches. Can't be gamed by
# over-segmenting (hurts precision) or under-segmenting (hurts recall) the way coverage
# alone can -- see chat history for the full reasoning.
#
# batch_rare_cross_batch_homog_mean added -- same formula/denominator as homogeneity
# (fraction of a rare cell's metacell that shares its label), but the numerator only
# counts same-label metacell-mates from a DIFFERENT batch. Coverage/homogeneity/F1
# can all be satisfied by a method giving each batch's rare cells their own small
# same-batch-only cluster -- exactly the "no co-clustering" failure mode
# harmony_notes.md describes. This metric can't be gamed that way: same-batch
# same-label mates never count toward it, so it directly tests whether rare types
# are actually grouped WITH their cross-batch counterparts, not just clustered.
#
# df_rare.index deduplicated below as a defensive guard -- extract_model_key() strips
# dataset-specific tokens (ds-panc/NP220/cvae_e50/v31/...) before substring-matching,
# which risks >1 of the 3 scProto keywords independently resolving for the same
# dataset and producing duplicate (dataset, 'scProto') rows -- pandas' unstack() then
# raises ValueError on the duplicate MultiIndex. Deduplicating (keep='first') is a
# no-op if it turns out not to be needed, and prevents the crash either way.
# model_keywords: reuses MODEL_KEYWORDS built once in the Table 1 cell above (same
# scProto canonical runs + SEACells(PCA) + one {SEACells, Leiden} entry per method in
# CORRECTION_METHODS) -- no longer duplicated here.
df_rare = rare_celltype_purity_table(RNA_SEQ_DATASETS, model_keywords=MODEL_KEYWORDS, verbose=True)

_dupe_mask = df_rare.index.duplicated(keep='first')
if _dupe_mask.any():
    print(f"WARNING: dropped {_dupe_mask.sum()} duplicate row(s) from df_rare: "
          f"{df_rare.index[_dupe_mask].tolist()}")
    df_rare = df_rare[~_dupe_mask]

show_table(
    df_rare,
    metrics=[
        'batch_rare_coverage_mean', 'batch_rare_recall_macro_mean',
        'batch_rare_precision_macro_mean', 'batch_rare_homogeneity_mean',
        'batch_rare_cross_batch_homog_mean', 'batch_rare_f1_macro_mean',
    ],
    dataset_display_names=dataset_display_names,
)

  [scProto|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] resolving run dir ...
  [SEACells (scVI (Gaussian))|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] run dir resolved (0.0s)
  [Leiden (scVI (Gaussian))|pancreas] resolving run dir ...
  [SEACells (Harmony)|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] reading umap_cells.csv ...
  [Leiden (Harmony)|pancreas] resolving run dir ...  [scProto|lung] resolving run dir ...

  [SEACells (PCA)|lung] resolving run dir ...
  [SEACells (PCA)|lung] run dir resolved (0.0s)
  [SEACells (PCA)|lung] reading umap_cells.csv ...
  [scProto|lung] run dir resolved (0.0s)
  [scProto|lung] reading umap_cells.csv ...
  [SEACells (scVI (Gaussian))|pancreas] run dir resolved (0.0s)
  [Leiden (scVI (Gaussian))|pancreas] run dir resolved (0.0s)  [scProto|pancreas] run dir resolved (0.0s)
  [SEACells (Harmony)|pancreas] run dir resolved (0.0s)

  [SEACells (scVI (Gaussian))|pancreas] reading umap_cells.csv ...
  [Leiden

In [16]:
# --- Significance test on the rare-cell F1/homogeneity table above ---
# Paired one-sided Mann-Whitney U (H1: scProto > other), Bonferroni-corrected across
# the non-ref methods compared within each (dataset, metric) group -- same convention
# already used for the spatial Fig. 3 significance test elsewhere in this codebase.
# Needs df_rare's raw '_batch_rare_f1_macro_per_batch' / '_batch_rare_homogeneity_per_batch'
# list columns (kept by default, not the display-filtered show_table view above) --
# uses the same df_rare computed in rareTableCell01, so run that cell first.
from interpretable_ssl.evaluation.paper_figures import rare_metric_significance

df_sig = rare_metric_significance(
    df_rare,
    ref_name='scProto',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
        '_batch_rare_cross_batch_homog_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)
df_sig

,dataset,metric,method,k,n,median,mean,std,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,0.032643,0.163214,ns
2,Pancreas,batch_rare_f1_macro,SEACells (scVI (Gaussian)),220,8,0.080208,0.177817,0.305975,0.006657,0.033283,*
3,Pancreas,batch_rare_f1_macro,Leiden (scVI (Gaussian)),220,8,0.000000,0.136982,0.296791,0.004122,0.020610,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony),220,8,0.295521,0.316501,0.159421,0.014064,0.070319,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony),220,8,0.133361,0.223580,0.171156,0.002331,0.011655,*
6,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,0.010334,0.051671,ns
8,Pancreas,batch_rare_homogeneity,SEACells (scVI (Gaussian)),220,8,0.073982,0.188286,0.298413,0.005206,0.026030,*
9,Pancreas,batch_rare_homogeneity,Leiden (scVI (Gaussian)),220,8,0.118845,0.214499,0.284644,0.005206,0.026030,*


In [17]:
# --- PAIRED version of the test above (recommended) ---
# The test above (rare_metric_significance) is UNPAIRED Mann-Whitney U, despite its
# own docstring calling it "paired". Since scProto and each baseline are scored on
# the exact same batches, a genuinely PAIRED test (Wilcoxon signed-rank on the
# per-batch differences) removes shared batch-to-batch variance from the comparison
# and has more power on this exact same data -- no new runs needed. See
# rare_metric_significance_paired's docstring in paper_figures.py for the full
# reasoning. Also adds n_wins: how many of the n batches scProto's value beats the
# baseline's outright -- easy to read even when p_adj stays 'ns'.
from interpretable_ssl.evaluation.paper_figures import rare_metric_significance_paired

df_sig_paired = rare_metric_significance_paired(
    df_rare,
    ref_name='scProto',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
        '_batch_rare_cross_batch_homog_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)
df_sig_paired


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,6.0,0.039062,0.195312,ns
2,Pancreas,batch_rare_f1_macro,SEACells (scVI (Gaussian)),220,8,0.080208,0.177817,0.305975,7.0,0.007812,0.039062,*
3,Pancreas,batch_rare_f1_macro,Leiden (scVI (Gaussian)),220,8,0.000000,0.136982,0.296791,8.0,0.003906,0.019531,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony),220,8,0.295521,0.316501,0.159421,7.0,0.011719,0.058594,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony),220,8,0.133361,0.223580,0.171156,8.0,0.003906,0.019531,*
6,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,8.0,0.003906,0.019531,*
8,Pancreas,batch_rare_homogeneity,SEACells (scVI (Gaussian)),220,8,0.073982,0.188286,0.298413,7.0,0.007812,0.039062,*
9,Pancreas,batch_rare_homogeneity,Leiden (scVI (Gaussian)),220,8,0.118845,0.214499,0.284644,7.0,0.007812,0.039062,*


In [18]:
# --- Embedding-level rare-cell kNN purity: was structure already lost before any
# clustering ran? (no clustering involved in this check) ---
rows = []
for ds_id in RNA_SEQ_DATASETS:
    path = os.path.join(get_dataset_model_dir(ds_id), f'rare_knn_purity_{ds_id}.json')
    if not os.path.exists(path):
        continue
    d = json.load(open(path))
    rows.append({'dataset': ds_id, **{m: v['mean_purity'] for m, v in d.items()}})

pd.DataFrame(rows).set_index('dataset')

,raw_pca,harmony,scvi_gauss,stage1z,scvi
dataset,,,,,
pancreas,0.499371,0.239623,0.147170,0.579874,0.45283
lung,0.912445,0.414134,0.456534,0.696960,NaN
pbmc-immune,0.776144,0.564410,0.433785,0.697679,NaN


## Embedding-only rare-cell affinity purity (no downstream clustering)

F1/homogeneity in the rare-cell table above is a *pipeline*-level metric -- it bundles
embedding quality with whichever downstream clustering algorithm ran on top (SEACells'
archetypal search vs. Leiden's modularity optimization vs. scProto's own trained,
joint assignment -- three different mechanisms, each with independent rare-state
handling behavior, sitting on top of one shared metric). This section isolates the
embedding: for each method's raw embedding, build one ARBF affinity graph directly on
it (no SEACells/Leiden/prototype-assignment step involved), then for each
locally-rare-type cell score what fraction of its total affinity MASS goes to
same-type cells. Also includes a dimensionality-MATCHED Harmony/PCA control at
scProto's own latent dimension, since scProto's latent (d=8) vs. Harmony's usual
embedding (d=50) isn't an apples-to-apples comparison on its own.

**No separate step to run here** -- `compute_and_save_embedding_affinity_purity`
(`interpretable_ssl/evaluation/batch_correct_baselines.py`) now runs automatically at
the end of `run_all_baselines_for_dataset`, so it already ran as part of the
`Run: Pancreas` / `Run: Lung` / `Run: PBMC (Immune)` cells above and saved its results
to `rare_affinity_purity_{ds_id}.json` per dataset. The cell below just loads and
displays those files -- rerun the `Run:` cells above if you need fresh numbers.

In [19]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

df_affinity_purity = load_and_compare_affinity_purity(
    RNA_SEQ_DATASETS, dataset_display_names=dataset_display_names,
)
df_affinity_purity


,dataset,method,dim,n,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,Raw PCA (uncorrected),50,8,0.385,0.164,7.0,0.0742,0.3711,ns
1,Pancreas,scProto,8,8,0.454,0.184,NaN,NaN,NaN,NaN
2,Pancreas,scVI (Gaussian),8,8,0.261,0.267,7.0,0.0078,0.0391,*
3,Pancreas,Harmony,8,8,0.297,0.129,7.0,0.0195,0.0977,ns
4,Pancreas,scPoli (Stage-1),8,8,0.512,0.162,3.0,0.8750,1.0000,ns
5,Pancreas,scVI,8,8,0.365,0.175,8.0,0.0039,0.0195,*
6,Lung,Raw PCA (uncorrected),50,15,0.559,0.194,11.0,0.0240,0.0958,ns
7,Lung,scProto,8,15,0.586,0.208,NaN,NaN,NaN,NaN
8,Lung,scVI (Gaussian),8,15,0.424,0.138,14.0,0.0002,0.0006,***
9,Lung,Harmony,8,15,0.423,0.095,13.0,0.0062,0.0249,*


## scIB integration metrics (e9Ho Q3)

> "Report the standard scIB metrics. Will you add standard metrics like ARI, NMI, ASW,
> kBET, iLISI, and graph-connectivity on the datasets?"

`get_scib()` and `get_all_embeddings_for_scib()`
(`interpretable_ssl/evaluation/metric_helpers/embedding_metrics.py`) wrap
`scib_metrics.benchmark.Benchmarker`, which computes the full standard
bio-conservation + batch-correction battery (isolated-label ASW/F1, NMI, ARI,
silhouette batch, iLISI, kBET, graph connectivity) directly from a set of `obsm`
embeddings -- this was already implemented elsewhere in the codebase but never run on
these three datasets/methods together until now.

`get_all_embeddings_for_scib()` reuses embeddings already computed above (Harmony/scVI
caches from `run_all_baselines_for_dataset`, run earlier in this notebook) plus
scProto's own Stage-2 latent, reloaded from its existing trained checkpoint (no
retraining). Run the `Run: Pancreas` / `Run: Lung` / `Run: PBMC (Immune)` cells above at
least once first so the Harmony/scVI caches exist on disk.


In [20]:
# from interpretable_ssl.evaluation.metric_helpers.embedding_metrics import (
#     get_scib, get_all_embeddings_for_scib,
# )

# EMBEDDING_KEYS_FOR_SCIB = ['X_pca', 'X_stage1z', 'X_harmony', 'X_scvi', 'X_scproto']
# EMBEDDING_DISPLAY_NAMES = {
#     'X_pca': 'Raw PCA (uncorrected)',
#     'X_stage1z': 'scPoli (Stage-1)',
#     'X_harmony': 'Harmony',
#     'X_scvi': 'scVI',
#     'X_scproto': 'scProto',
# }

# # This can take a while per dataset -- kBET/iLISI neighbor graphs are not cheap on
# # 16-30k cells x up to 5 embeddings at once. Expect several minutes per dataset.
# scib_results = {}
# for ds_id in RNA_SEQ_DATASETS:
#     print(f"\n=== [{ds_id}] computing scIB metrics ===")
#     lk = DATASETS[ds_id]['label_key']
#     bk = DATASETS[ds_id].get('batch_key')

#     ad_scib = get_all_embeddings_for_scib(ds_id)
#     keys_present = [k for k in EMBEDDING_KEYS_FOR_SCIB if k in ad_scib.obsm]
#     missing_keys = [k for k in EMBEDDING_KEYS_FOR_SCIB if k not in ad_scib.obsm]
#     if missing_keys:
#         print(f"[{ds_id}] proceeding without: {missing_keys}")

#     scib_df = get_scib(ad_scib, keys_present, bk, lk)
#     if scib_df is None:
#         print(f"[{ds_id}] get_scib returned None (single batch?) -- skipping.")
#         continue

#     scib_df = scib_df.rename(index=EMBEDDING_DISPLAY_NAMES)
#     save_path = os.path.join(get_dataset_model_dir(ds_id), 'scib_metrics.csv')
#     scib_df.to_csv(save_path)
#     print(f"[{ds_id}] scIB metrics saved to {save_path}")
#     scib_results[ds_id] = scib_df

# scib_results


In [21]:
# for ds_id, df in scib_results.items():
#     print(f"\n=== {dataset_display_names.get(ds_id, ds_id)} ===")
#     display(df)


# significant tests

In [22]:
# --- Significance test for the rare-cell metrics above (F1, homogeneity) ---
#
# Reviewer e9Ho (Weakness 7): "Tables 1-2 report no significance test ... some
# 'wins' overlap in std." This is a PAIRED one-sided Wilcoxon signed-rank test
# (ref > other) -- scProto and each baseline are scored on the exact same batches
# for a given dataset, so pairing removes shared batch-to-batch variance (e.g.
# "batch 3 is just noisier for everyone") and isolates just the
# scProto-vs-baseline difference (see rare_metric_significance_paired's docstring
# in paper_figures.py for the full reasoning). An earlier UNPAIRED Mann-Whitney U
# version of this same test sits right after the rare-cell table above -- kept
# there for comparison, this cell is the recommended one to cite.
#
# Bonferroni-corrected across the non-ref methods compared within each dataset.
# Runs directly on the per-batch F1/homogeneity arrays underlying df_rare above --
# no retraining needed.
#
# Only same-K comparisons are made: a baseline whose total metacell count K
# differs from scProto's for that dataset is dropped before testing (not
# compared at all), per the paper's own protocol ("All baselines are
# configured to produce the same number of metacells K as scProto") --
# printed below as 'Skipped (K mismatch ...)' if any run doesn't match.
#
# scProto is the reference (alternative='greater'): p_adj small => scProto's
# median is significantly higher than that baseline's, not just numerically
# ahead. 'ns' means the two are not significantly separated at alpha=0.05 after
# correction -- an honest result to report, not a bug. n_wins: out of n batches,
# how many scProto's value beats the baseline's outright -- easy to read
# regardless of p_adj, and stays informative even when 'ns'.

from interpretable_ssl.evaluation.paper_figures import rare_metric_significance_paired

sig_df = rare_metric_significance_paired(
    df_rare,
    ref_name='scProto',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
        '_batch_rare_cross_batch_homog_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)

for metric_name in sig_df['metric'].unique():
    print(f"=== {metric_name}: scProto vs. each same-K baseline, PAIRED one-sided "
          f"Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===")
    sub = sig_df[sig_df['metric'] == metric_name].copy()
    sub['cell'] = sub.apply(
        lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
        else f"{r['median']:.3f} (K={r['k']}, n={r['n']}, wins={r.get('n_wins', '?')}/{r['n']})  "
             f"{r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
        axis=1,
    )
    display(sub.pivot(index='method', columns='dataset', values='cell'))

sig_df


=== batch_rare_f1_macro: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.766 (K=300, n=5, wins=4.0/5) ns p_adj=0.312","0.282 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.133 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
Leiden (scVI (Gaussian)),"0.676 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.299 (K=300, n=15, wins=15.0/15) *** p_adj=...","0.000 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
SEACells (Harmony),"0.812 (K=300, n=5, wins=4.0/5) ns p_adj=0.312","0.348 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.296 (K=220, n=8, wins=7.0/8) ns p_adj=0.0586"
SEACells (PCA),"0.923 (K=300, n=5, wins=2.0/5) ns p_adj=1","0.498 (K=300, n=15, wins=11.0/15) ** p_adj=0...","0.373 (K=220, n=8, wins=6.0/8) ns p_adj=0.195"
SEACells (scVI (Gaussian)),"0.652 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.342 (K=300, n=15, wins=15.0/15) *** p_adj=...","0.080 (K=220, n=8, wins=7.0/8) * p_adj=0.0391"
scProto,"0.897 (K=294, n=5) [ref]","0.611 (K=298, n=15) [ref]","0.438 (K=219, n=8) [ref]"


=== batch_rare_homogeneity: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.707 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.345 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.220 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
Leiden (scVI (Gaussian)),"0.538 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.327 (K=300, n=15, wins=15.0/15) *** p_adj=...","0.119 (K=220, n=8, wins=7.0/8) * p_adj=0.0391"
SEACells (Harmony),"0.804 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.408 (K=300, n=15, wins=14.0/15) * p_adj=0....","0.269 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
SEACells (PCA),"0.815 (K=300, n=5, wins=3.0/5) ns p_adj=1","0.495 (K=300, n=15, wins=13.0/15) * p_adj=0....","0.369 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
SEACells (scVI (Gaussian)),"0.582 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.385 (K=300, n=15, wins=14.0/15) *** p_adj=...","0.074 (K=220, n=8, wins=7.0/8) * p_adj=0.0391"
scProto,"0.856 (K=294, n=5) [ref]","0.586 (K=298, n=15) [ref]","0.522 (K=219, n=8) [ref]"


=== batch_rare_cross_batch_homog: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.373 (K=300, n=5, wins=2.0/5) ns p_adj=1","0.341 (K=300, n=15, wins=13.0/15) * p_adj=0....","0.184 (K=220, n=8, wins=6.0/8) ns p_adj=0.195"
Leiden (scVI (Gaussian)),"0.344 (K=300, n=5, wins=4.0/5) ns p_adj=0.17","0.318 (K=300, n=15, wins=14.0/15) *** p_adj=...","0.064 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
SEACells (Harmony),"0.305 (K=300, n=5, wins=3.0/5) ns p_adj=0.781","0.399 (K=300, n=15, wins=14.0/15) * p_adj=0....","0.210 (K=220, n=8, wins=6.0/8) ns p_adj=0.371"
SEACells (PCA),"0.029 (K=300, n=5, wins=3.0/5) ns p_adj=0.36","0.416 (K=300, n=15, wins=12.0/15) ** p_adj=0...","0.212 (K=220, n=8, wins=5.0/8) ns p_adj=0.625"
SEACells (scVI (Gaussian)),"0.347 (K=300, n=5, wins=4.0/5) ns p_adj=0.17","0.377 (K=300, n=15, wins=13.0/15) *** p_adj=...","0.041 (K=220, n=8, wins=8.0/8) * p_adj=0.0195"
scProto,"0.407 (K=294, n=5) [ref]","0.510 (K=298, n=15) [ref]","0.259 (K=219, n=8) [ref]"


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,6.0,0.039062,0.195312,ns
2,Pancreas,batch_rare_f1_macro,SEACells (scVI (Gaussian)),220,8,0.080208,0.177817,0.305975,7.0,0.007812,0.039062,*
3,Pancreas,batch_rare_f1_macro,Leiden (scVI (Gaussian)),220,8,0.000000,0.136982,0.296791,8.0,0.003906,0.019531,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony),220,8,0.295521,0.316501,0.159421,7.0,0.011719,0.058594,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony),220,8,0.133361,0.223580,0.171156,8.0,0.003906,0.019531,*
6,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,8.0,0.003906,0.019531,*
8,Pancreas,batch_rare_homogeneity,SEACells (scVI (Gaussian)),220,8,0.073982,0.188286,0.298413,7.0,0.007812,0.039062,*
9,Pancreas,batch_rare_homogeneity,Leiden (scVI (Gaussian)),220,8,0.118845,0.214499,0.284644,7.0,0.007812,0.039062,*


In [23]:
# --- Significance test for Table 1 (modularity, batch entropy, purity) ---
#
# Reviewer e9Ho (Weakness 7): "Tables 1-2 report no significance test." The cell
# above covers Table 2 (F1, homogeneity); this is the Table 1 counterpart --
# same one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected across
# baselines per dataset, same K-matching (realized metacell count within 5%
# tolerance, since a few empty prototypes/archetypes are normal even at
# identical configured K -- see rareSignificanceCell01 above for why exact
# equality is the wrong bar).
#
# Reads modularity_per_batch.csv / purity_per_mc.csv / batch_entropy_per_mc.csv
# directly from each run's saved directory (the same files load_task1_multi
# already reduces to the mean/std shown in Table 1 above) -- no retraining
# needed. Reuses the shared MODEL_KEYWORDS dict built in the Table 1 cell above,
# so method names/canonical runs are identical across every table and test.

from interpretable_ssl.evaluation.paper_figures import graph_batch_significance

sig_df_table1 = graph_batch_significance(
    RNA_SEQ_DATASETS,
    MODEL_KEYWORDS,
    ref_name='scProto',
    dataset_display_names=dataset_display_names,
)

for metric_name in sig_df_table1['metric'].unique():
    print(f"=== {metric_name}: scProto vs. each same-K baseline, one-sided "
          f"Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===")
    sub = sig_df_table1[sig_df_table1['metric'] == metric_name].copy()
    sub['cell'] = sub.apply(
        lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
        else f"{r['median']:.3f} (K={r['k']}, n={r['n']})  {r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
        axis=1,
    )
    display(sub.pivot(index='method', columns='dataset', values='cell'))

sig_df_table1


=== modularity_per_batch: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.250 (K=300, n=5) * p_adj=0.0198","0.403 (K=300, n=16) *** p_adj=4.66e-06","0.614 (K=220, n=9) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.344 (K=300, n=5) ns p_adj=0.0794","0.300 (K=300, n=16) *** p_adj=3.86e-06","0.340 (K=220, n=9) ** p_adj=0.00272"
SEACells (Harmony),"0.551 (K=300, n=5) ns p_adj=0.0794","0.625 (K=300, n=16) *** p_adj=0.000798","0.566 (K=220, n=9) ns p_adj=0.159"
SEACells (PCA),"0.569 (K=300, n=5) ns p_adj=0.139","0.671 (K=300, n=16) ns p_adj=1","0.658 (K=220, n=9) ns p_adj=1"
SEACells (scVI (Gaussian)),"0.657 (K=300, n=5) ns p_adj=1","0.141 (K=300, n=16) *** p_adj=3.86e-06","0.720 (K=220, n=9) ns p_adj=1"
scProto,"0.620 (K=294, n=5) [ref]","0.669 (K=298, n=16) [ref]","0.621 (K=219, n=9) [ref]"


=== purity_per_mc: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.855 (K=300, n=300) *** p_adj=2.58e-17","0.841 (K=300, n=300) *** p_adj=1.16e-12","1.000 (K=220, n=220) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.799 (K=300, n=300) *** p_adj=8.93e-35","0.831 (K=300, n=300) *** p_adj=6.64e-12","1.000 (K=220, n=220) ns p_adj=1"
SEACells (Harmony),"0.883 (K=300, n=300) *** p_adj=4.41e-15","0.832 (K=300, n=300) *** p_adj=8.01e-12","0.967 (K=220, n=220) *** p_adj=0.000186"
SEACells (PCA),"0.965 (K=300, n=300) *** p_adj=8.99e-06","0.977 (K=300, n=300) ns p_adj=1","0.993 (K=220, n=220) ns p_adj=1"
SEACells (scVI (Gaussian)),"0.906 (K=300, n=300) *** p_adj=4.33e-15","0.944 (K=300, n=300) *** p_adj=0.000159","0.981 (K=220, n=220) ns p_adj=0.0654"
scProto,"1.000 (K=294, n=294) [ref]","0.996 (K=298, n=298) [ref]","1.000 (K=219, n=219) [ref]"


=== batch_entropy_per_mc: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"1.071 (K=300, n=300) ns p_adj=1","1.350 (K=300, n=300) ns p_adj=1","1.327 (K=220, n=220) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.769 (K=300, n=300) ns p_adj=1","1.122 (K=300, n=300) ns p_adj=1","0.805 (K=220, n=220) ns p_adj=1"
SEACells (Harmony),"0.726 (K=300, n=300) ns p_adj=1","1.526 (K=300, n=300) ns p_adj=1","1.339 (K=220, n=220) ns p_adj=1"
SEACells (PCA),"-0.000 (K=300, n=300) ns p_adj=1","0.218 (K=300, n=300) ns p_adj=1","-0.000 (K=220, n=220) *** p_adj=7.98e-06"
SEACells (scVI (Gaussian)),"0.434 (K=300, n=300) ns p_adj=1","1.116 (K=300, n=300) ns p_adj=1","0.882 (K=220, n=220) ns p_adj=1"
scProto,"-0.000 (K=294, n=294) [ref]","0.500 (K=298, n=298) [ref]","0.214 (K=219, n=219) [ref]"


,dataset,metric,method,k,n,median,mean,std,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,6.214564e-01,0.601234,0.083385,NaN,NaN,NaN
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,6.578915e-01,0.673906,0.050910,9.739706e-01,1.000000e+00,ns
2,Pancreas,modularity_per_batch,SEACells (scVI (Gaussian)),220,9,7.202421e-01,0.725213,0.038824,9.994569e-01,1.000000e+00,ns
3,Pancreas,modularity_per_batch,Leiden (scVI (Gaussian)),220,9,3.404861e-01,0.352439,0.100534,5.431233e-04,2.715617e-03,**
4,Pancreas,modularity_per_batch,SEACells (Harmony),220,9,5.663669e-01,0.567150,0.017709,3.184489e-02,1.592244e-01,ns
5,Pancreas,modularity_per_batch,Leiden (Harmony),220,9,6.143183e-01,0.612469,0.021231,3.619660e-01,1.000000e+00,ns
6,Pancreas,purity_per_mc,scProto,219,219,1.000000e+00,0.912529,0.160439,NaN,NaN,NaN
7,Pancreas,purity_per_mc,SEACells (PCA),220,220,9.931271e-01,0.956404,0.090829,4.277007e-01,1.000000e+00,ns
8,Pancreas,purity_per_mc,SEACells (scVI (Gaussian)),220,220,9.813068e-01,0.930490,0.128800,1.307339e-02,6.536697e-02,ns
9,Pancreas,purity_per_mc,Leiden (scVI (Gaussian)),220,220,1.000000e+00,0.919241,0.163201,5.001637e-01,1.000000e+00,ns


In [24]:
# --- PAIRED significance test for Table 1's modularity (recommended for this metric) ---
# Only modularity_per_batch is pairable here -- batch is a shared, method-independent
# unit (see graph_batch_significance_paired's docstring in paper_figures.py).
# purity_per_mc and batch_entropy_per_mc stay unpaired (Mann-Whitney U, the cell
# above) -- a metacell is a method-specific output (different clustering methods
# produce different numbers of metacells with different compositions), so there is
# no natural cross-method correspondence to pair on. That's the statistically
# correct test for those two, not a limitation to fix.

from interpretable_ssl.evaluation.paper_figures import graph_batch_significance_paired

sig_df_table1_paired = graph_batch_significance_paired(
    RNA_SEQ_DATASETS,
    MODEL_KEYWORDS,
    ref_name='scProto',
    dataset_display_names=dataset_display_names,
)

sub = sig_df_table1_paired.copy()
sub['cell'] = sub.apply(
    lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
    else f"{r['median']:.3f} (K={r['k']}, n={r['n']}, wins={r.get('n_wins', '?')}/{r['n']})  "
         f"{r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
    axis=1,
)
display(sub.pivot(index='method', columns='dataset', values='cell'))

sig_df_table1_paired


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.250 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.403 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.614 (K=220, n=9, wins=6.0/9) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.344 (K=300, n=5, wins=4.0/5) ns p_adj=0.312","0.300 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.340 (K=220, n=9, wins=8.0/9) * p_adj=0.0195"
SEACells (Harmony),"0.551 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.625 (K=300, n=16, wins=14.0/16) ** p_adj=0...","0.566 (K=220, n=9, wins=7.0/9) ns p_adj=0.625"
SEACells (PCA),"0.569 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.671 (K=300, n=16, wins=9.0/16) ns p_adj=1","0.658 (K=220, n=9, wins=0.0/9) ns p_adj=1"
SEACells (scVI (Gaussian)),"0.657 (K=300, n=5, wins=1.0/5) ns p_adj=1","0.141 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.720 (K=220, n=9, wins=0.0/9) ns p_adj=1"
scProto,"0.620 (K=294, n=5) [ref]","0.669 (K=298, n=16) [ref]","0.621 (K=219, n=9) [ref]"


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,0.621456,0.601234,0.083385,NaN,NaN,NaN,NaN
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,0.657892,0.673906,0.050910,0.0,1.000000,1.000000,ns
2,Pancreas,modularity_per_batch,SEACells (scVI (Gaussian)),220,9,0.720242,0.725213,0.038824,0.0,1.000000,1.000000,ns
3,Pancreas,modularity_per_batch,Leiden (scVI (Gaussian)),220,9,0.340486,0.352439,0.100534,8.0,0.003906,0.019531,*
4,Pancreas,modularity_per_batch,SEACells (Harmony),220,9,0.566367,0.567150,0.017709,7.0,0.125000,0.625000,ns
5,Pancreas,modularity_per_batch,Leiden (Harmony),220,9,0.614318,0.612469,0.021231,6.0,0.544922,1.000000,ns
6,Lung,modularity_per_batch,scProto,298,16,0.669390,0.664444,0.022871,NaN,NaN,NaN,NaN
7,Lung,modularity_per_batch,SEACells (PCA),300,16,0.671340,0.673784,0.037789,9.0,0.628220,1.000000,ns
8,Lung,modularity_per_batch,SEACells (scVI (Gaussian)),300,16,0.140850,0.140271,0.023944,16.0,0.000015,0.000076,***
9,Lung,modularity_per_batch,Leiden (scVI (Gaussian)),300,16,0.300442,0.276332,0.087973,16.0,0.000015,0.000076,***
